# Check a draft's claims

A claim is an HTML comment beside the prose it describes. `attest claims`
re-derives the number from the experiment runs on disk and gives a verdict per
claim. This notebook checks the worked example `speech-distill/FINDINGS.md`,
in which three claims are **deliberately wrong**, and shows what each kind of
miss means.

It runs offline (no model server) against a temporary ledger.

Copy this notebook before editing it: the copy that ships with the skill is
overwritten whenever attestation updates it.

In [1]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# Everything this notebook writes goes to a throwaway folder and a throwaway
# ledger. Your real database and workspace are never read or changed.
WORK = Path(tempfile.mkdtemp(prefix="attest-demo-"))
os.environ["ATTEST_DB"] = str(WORK / "demo.db")
HOME = WORK / "hermes"
os.environ["HERMES_HOME"] = str(HOME)
# Point these at the temp folder rather than just unsetting them: every `attest`
# call reloads a checkout's .env, which would put an unset variable back.
os.environ["LEDGER_METRIC_DIRECTION_FILE"] = str(HOME / "metric_direction.toml")
os.environ["LEDGER_CORPUS_FILE"] = str(HOME / "corpus.toml")
os.environ["RESEARCH_ROOT"] = str(WORK / "workspace")


def _find_attest():
    found = shutil.which("attest", path=str(Path(sys.executable).parent)) or shutil.which("attest")
    if found is None:
        raise RuntimeError(
            "`attest` is not on PATH: run this notebook in the attestation environment"
        )
    return found


_ATTEST = _find_attest()


def attest(command, cwd):
    """Run `attest <command>` offline in `cwd`, print what it prints, return its exit code."""
    print("$ attest " + command)
    done = subprocess.run(
        [_ATTEST, *shlex.split(command)], cwd=cwd, capture_output=True, text=True, timeout=60
    )
    text = (done.stdout + done.stderr).rstrip()
    print(text.replace(str(WORK), "<tmp>").replace(str(Path.home()), "~"))
    print(f"[exit {done.returncode}]")
    return done.returncode


def _run(tag, wer, val_loss):
    record = {"tag": tag, "corpus": "librispeech-100h", "wer": wer, "val_loss": val_loss}
    return json.dumps({**record, "n_records": 2620})


FALLBACK = {
    "speech-distill/results/kdsweep_baseline.json": _run("kdsweep_baseline", 0.0731, 2.4),
    "speech-distill/results/kdsweep_t2.json": _run("kdsweep_t2", 0.0688, 2.3),
    "speech-distill/results/kdsweep_t4.json": _run("kdsweep_t4", 0.0642, 2.18),
    "speech-distill/results/kdsweep_t4b.json": _run("kdsweep_t4b", 0.0659, 2.2),
    "speech-distill/FINDINGS.md": """# Distillation notes

The baseline reaches a word error rate of 0.0731.
<!-- claim: speech-distill/kdsweep_baseline metric=wer value=0.0731 -->

Temperature 4 is better, at 0.0642.
<!-- claim: speech-distill/kdsweep_t4 metric=wer value=0.0642 -->

Temperature 2 gives 0.0701 (the run records a different number).
<!-- claim: speech-distill/kdsweep_t2 metric=wer value=0.0701 -->

Temperature 8 gives 0.0600 (there is no such run).
<!-- claim: speech-distill/kdsweep_t8 metric=wer value=0.0600 -->

A claim with no metric at all.
<!-- claim: speech-distill/kdsweep_baseline value=0.0731 -->

Validation loss ended at 2.18 and the model has 41.3M parameters.
""",
}


def find_example_workspace():
    """examples/workspace from a repo checkout, or None. ATTEST_EXAMPLE_WORKSPACE
    points at one explicitly; the value `none` skips the search."""
    override = os.environ.get("ATTEST_EXAMPLE_WORKSPACE")
    if override == "none":
        return None
    if override:
        return Path(override)
    import attestation

    for base in (Path.cwd(), Path(attestation.__file__).resolve().parent):
        for folder in (base, *base.parents):
            candidate = folder / "examples" / "workspace"
            if (candidate / "speech-distill" / "FINDINGS.md").is_file():
                return candidate
    return None


WORKSPACE = WORK / "workspace"
SOURCE = find_example_workspace()
if SOURCE is not None:
    shutil.copytree(SOURCE, WORKSPACE)
    print("Using the example workspace from the attestation repo (copied to a temporary folder).")
else:
    for name, text in FALLBACK.items():
        target = WORKSPACE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text)
    print("The repo's examples/workspace is not on this machine, so this notebook")
    print("wrote a small stand-in with the same shape (also in a temporary folder).")

Using the example workspace from the attestation repo (copied to a temporary folder).


## 1. Read the runs into the ledger

Claims are checked against the ledger, so it must hold the runs first. Against
an empty ledger every claim would come back `unsupported`, which would mean
"nothing was scanned", not "the draft is wrong".

In [2]:
attest("runs scan --root .", cwd=WORKSPACE)

$ attest runs scan --root .


  retrieval-ablation           5 run(s)
  speech-distill               4 run(s)
9 run(s) across 2 project(s)
[exit 0]


0

## 2. Check every claim in the draft

The same check from Python, as a table: one row per claim.

In [3]:
import contextlib

from IPython.display import Markdown, display

from attestation import claims
from attestation.db import get_db

os.chdir(WORKSPACE)  # the ledger stores source paths relative to the scan root
with contextlib.closing(get_db(Path(os.environ["ATTEST_DB"]))) as conn:
    result = claims.check(conn, WORKSPACE / "speech-distill" / "FINDINGS.md")

rows = ["| line | claim | verdict | why |", "|---:|---|---|---|"]
for v in result["verdicts"]:
    claim = f"`{v.claim.run} {v.claim.metric}={v.claim.value:g}`"
    rows.append(f"| {v.claim.line} | {claim} | **{v.verdict}** | {v.message} |")
for problem in result["malformed"]:
    where, _, reason = problem.rpartition(": ")
    rows.append(f"| {where.rpartition(':')[2]} | (cannot be read) | **malformed** | {reason} |")
display(Markdown("\n".join(rows)))
counts = {k.value: n for k, n in result["counts"].items()}
print(counts, "plus", len(result["malformed"]), "malformed")

| line | claim | verdict | why |
|---:|---|---|---|
| 18 | `kdsweep_baseline wer=0.0731` | **supported** | wer=0.0731 in kdsweep_baseline |
| 21 | `kdsweep_t2 wer=0.0688` | **supported** | wer=0.0688 in kdsweep_t2 |
| 25 | `kdsweep_t4 wer=0.0642` | **supported** | wer=0.0642 in kdsweep_t4 |
| 28 | `kdsweep_t4 val_loss=2.18` | **supported** | val_loss=2.18 in kdsweep_t4 |
| 33 | `kdsweep_t2 wer=0.0701` | **contradicted** | document says 0.0701, run records 0.0688 (tolerance 1e-09) |
| 36 | `kdsweep_t8 wer=0.06` | **unsupported** | no run speech-distill/kdsweep_t8 in the ledger -- the claim may be true, but nothing here backs it |
| 46 | `kdsweep_t4b wer=0.0659` | **supported** | wer=0.0659 in kdsweep_t4b |
| 39 | (cannot be read) | **malformed** | missing metric |

{'supported': 5, 'contradicted': 1, 'unsupported': 1} plus 1 malformed


## 3. What the three misses mean

They are different problems and need different fixes. Never read one as the other.

| verdict | what it says | what you do |
|---|---|---|
| `contradicted` | a run **disagrees** with the number in the prose | the draft or the run is wrong; correct one of them |
| `unsupported` | **no run matches**: the run name does not exist (as here), or the run has no such metric | the claim may be true, but nothing backs it; find or record the run. It does not mean false |
| `malformed` | the annotation **cannot be read** (here: no `metric` field) | fix the annotation; it is reported rather than skipped so a claim cannot vanish from review |

`supported` means a run agrees within tolerance. Three more verdicts exist and
this draft has none of them: `ambiguous` (a wildcard matched several runs, or
one run holds the metric at several splits or steps and the claim does not say
which: add `split=` or `step=`), `stale` (the value matches a run whose file
changed after `as_of`) and `uncited` (a claim with `cite=` whose key no
configured bibliography source has).

## 4. The same check from the command line

`attest claims` exits 1 when a claim is contradicted, so it can gate a commit.
The exit code below is the check working, not a crash.

In [4]:
attest("claims speech-distill/FINDINGS.md", cwd=WORKSPACE)

$ attest claims speech-distill/FINDINGS.md


  malformed  speech-distill/FINDINGS.md:39: missing metric
  supported     FINDINGS.md:18               wer=0.0731  wer=0.0731 in kdsweep_baseline
  supported     FINDINGS.md:21               wer=0.0688  wer=0.0688 in kdsweep_t2
  supported     FINDINGS.md:25               wer=0.0642  wer=0.0642 in kdsweep_t4
  supported     FINDINGS.md:28               val_loss=2.18  val_loss=2.18 in kdsweep_t4
  contradicted  FINDINGS.md:33               wer=0.0701  document says 0.0701, run records 0.0688 (tolerance 1e-09)
  unsupported   FINDINGS.md:36               wer=0.06  no run speech-distill/kdsweep_t8 in the ledger -- the claim may be true, but nothing here backs it
  supported     FINDINGS.md:46               wer=0.0659  wer=0.0659 in kdsweep_t4b

7 claim(s): 1 contradicted, 5 supported, 1 unsupported
1 malformed
[exit 1]


1

## 5. Numbers no claim covers

A draft with zero contradicted claims can still assert numbers nothing backs.
`--coverage` lists them.

In [5]:
attest("claims speech-distill/FINDINGS.md --coverage", cwd=WORKSPACE)

$ attest claims speech-distill/FINDINGS.md --coverage


  uncovered  FINDINGS.md:13           41.3         41.3M parameters.
  uncovered  FINDINGS.md:23           12.2         Temperature 4 is better still, at 0.0642 -- a 12.2% relative
  uncovered  FINDINGS.md:44           0.0017       0.0017 -- larger than the gap between several of the arms ab
  uncovered  FINDINGS.md:50           41.3         The model has 41.3M parameters and trains for 20 epochs at a

5/9 number(s) covered by a claim across 1 file(s)
[exit 0]


0